In [8]:
import pandas as pd
import numpy as np
import emoji
import re
import os

In [9]:
load_path = 'full_data'

In [10]:
def clean_tweet(text):
    """Clean the tweet text by removing emojis, RTs, @users, URLs, and extra whitespace."""
    # Remove emojis
    text = emoji.replace_emoji(text, replace="")
    # Remove RT
    text = re.sub(r'RT\s+', '', text)
    # Remove @user
    text = re.sub(r'@\w+', '', text)
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|url', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def preprocess_data(df):
    """
    Preprocess the dataset by cleaning tweets and preparing labels.
    
    Parameters:
    df (DataFrame): Input dataframe with 'tweet' and 'sentiment' columns
    
    Returns:
    DataFrame: Cleaned dataframe with 'text' and 'label' columns
    """
    # Dropping unnecessary properties
    df_clean = df.drop(columns=['HITId','directness', 'annotator_sentiment', 'target', 'group'])
    
    # Dropping rows with null variables if they exist
    orig = df_clean.shape[0]
    df_clean.dropna(inplace=True)
    count = orig - df_clean.shape[0]
    
    print(count, "rows with null values were found")
    
    # Binary sentiments: 'normal' is 0, anything else is 1
    df_clean['sentiment'] = np.where(df_clean['sentiment'] == 'normal', 0, 1)
    
    # Getting rid of usernames, urls, RTs, and emojis
    df_clean['tweet'] = df_clean['tweet'].astype(str).apply(clean_tweet)
    
    # Rename columns
    df_clean.rename(columns={'tweet': 'text', 'sentiment': 'label'}, inplace=True)
    
    return df_clean

In [11]:
# Apply the preprocessing function
for filename in os.listdir(load_path):
    df = pd.read_csv(os.path.join(load_path,filename))
    df_clean = preprocess_data(df)
    df_clean.to_csv(os.path.join(load_path, filename), index=False)

KeyError: "['directness', 'annotator_sentiment', 'target', 'group'] not found in axis"